# Детекция ботов в диалогах. Марковский постпроцессор

Постановка задачи: исследование динамики турнирной сетки сессий, обучение скрытой марковской модели (HMM) и сглаживание вероятностей стекера.

Цели этапа:
- Извлечение честных OOF и тестовых предсказаний мета-стекера `TwoLevelStackingClassifier` из ClickHouse DWH.
- Оценка матрицы вероятностей перехода состояний матчинга участников в хронологической цепочке диалогов.
- Выполнение двунаправленного сглаживания вероятностей алгоритмом Forward-Backward HMM.
- Оптимизация веса блендинга марковского постпроцессора с базовыми предсказаниями стекера.
- Применение внутридиалоговой парной калибровки `PairwisePostprocessor` ($P_0 + P_1 \le 1.0$).
- Формирование финального соревновательного сабмишна и сохранение артефактов в MLflow.

In [1]:
import json
import os
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, log_loss, roc_auc_score

from project import Reproducibility
from project.core.config import config
from project.infrastructure.clickhouse import FeatureDWH
from project.models import (
    HMMPostprocessor,
    PairwisePostprocessor,
)
from project.tracking import ExperimentTrackerBuilder

2026/11/10 00:07:37 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


In [2]:
Reproducibility.seed(42)

## Загрузка турнирной сетки из ClickHouse DWH

Выгружаем хронологический порядок диалогов, целевую разметку и метаданные сессий из ClickHouse DWH.

In [3]:
client = mlflow.tracking.MlflowClient(config.mlflow_tracking_uri)
exp = client.get_experiment_by_name("bot-detection")
if not exp:
    raise RuntimeError("Эксперимент 'bot-detection' не найден в MLflow Tracking Server!")

dwh = FeatureDWH()
train_dialogs = dwh.raw.load_dialogues_dict("train")
test_dialogs = dwh.raw.load_dialogues_dict("test")
train_dialog_order = list(train_dialogs.keys())
test_dialog_order = list(test_dialogs.keys())
ytrain = dwh.raw.load_labels("train")
ytest = dwh.raw.load_labels("test")
groups = ytrain["dialog_id"].to_numpy()
y_train = ytrain["is_bot"].to_numpy().astype(int)

## Загрузка предсказаний мета-стекера TwoLevelStackingClassifier

Загружаем честные OOF и тестовые вероятности мета-стекера из ClickHouse DWH и MLflow Tracking.

In [4]:
# Загрузка честных OOF и Test предсказаний мета-стекера из ClickHouse DWH
runs = client.search_runs(
    exp.experiment_id,
    filter_string="attributes.run_name = 'logistic_meta_stacker' and attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
)
if not runs:
    runs = client.search_runs(
        exp.experiment_id,
        filter_string="tags.mlflow.runName = 'logistic_meta_stacker' and attributes.status = 'FINISHED'",
        order_by=["attributes.start_time DESC"],
    )
if not runs:
    raise RuntimeError("Не найден завершенный ран 'logistic_meta_stacker'!")

meta_stacker_run = runs[0]

meta_tr_df = dwh.predictions.load_predictions(run_id=meta_stacker_run.info.run_id, dataset_split="train_oof")
meta_te_df = dwh.predictions.load_predictions(run_id=meta_stacker_run.info.run_id, dataset_split="test")

# Выравниваем предсказания строго по порядку диалогов в ytrain и ytest
meta_tr_aligned = ytrain.select(["dialog_id", "participant_index"]).join(
    meta_tr_df.select(["dialog_id", "participant_index", "prob_bot"]),
    on=["dialog_id", "participant_index"],
    how="left",
)
meta_te_aligned = ytest.select(["dialog_id", "participant_index"]).join(
    meta_te_df.select(["dialog_id", "participant_index", "prob_bot"]),
    on=["dialog_id", "participant_index"],
    how="left",
)

oof_meta_raw = meta_tr_aligned["prob_bot"].to_numpy()
test_meta_raw = meta_te_aligned["prob_bot"].to_numpy()

pl.DataFrame(
    {
        "model": ["Level-2 Meta-Stacker Honest OOF (ClickHouse)"],
        "val_logloss": [round(float(log_loss(y_train, oof_meta_raw)), 5)],
        "val_roc_auc": [round(float(roc_auc_score(y_train, oof_meta_raw)), 5)],
    }
)

model,val_logloss,val_roc_auc
str,f64,f64
"""Level-2 Meta-Stacker Honest OO…",0.16084,0.98318


## Обучение марковского постпроцессора HMMPostprocessor

Обучаем трехсостоятельную модель турнирного матчинга:
- Состояние 0: (Человек, Человек)
- Состояние 1: (Бот, Человек)
- Состояние 2: (Человек, Бот)

Оцениваем матрицу переходов с регуляризацией Лапласа.

In [5]:
hmm_postprocessor = HMMPostprocessor(alpha_laplace=1.0, blend_weight=0.6, eps=0.005)
hmm_postprocessor.fit(
    dialog_order=train_dialog_order,
    y_true=y_train,
    participant_indices=ytrain["participant_index"].to_numpy(),
    dialog_ids=ytrain["dialog_id"].to_numpy(),
)
pl.DataFrame(
    {
        "metric": ["Initial P(Human, Human)", "Initial P(Bot, Human)", "Initial P(Human, Bot)"],
        "probability": [round(float(p), 4) for p in hmm_postprocessor.initial_probs_],
    }
)

metric,probability
str,f64
"""Initial P(Human, Human)""",0.184
"""Initial P(Bot, Human)""",0.496
"""Initial P(Human, Bot)""",0.3199


## Сравнительный анализ этапов постобработки

Сопоставляем качество сырого мета-стекера, эффекта марковского сглаживания HMM и парной нормализации диалогов.

In [6]:
pairwise_postprocessor = PairwisePostprocessor(max_pair_sum=1.0, eps=0.005)
oof_hmm_only = hmm_postprocessor.transform(
    probabilities=oof_meta_raw,
    dialog_ids=ytrain["dialog_id"].to_numpy(),
    participant_indices=ytrain["participant_index"].to_numpy(),
    dialog_order=train_dialog_order,
)
oof_pairwise_only = pairwise_postprocessor.transform(
    probabilities=oof_meta_raw,
    dialog_ids=ytrain["dialog_id"].to_numpy(),
    participant_indices=ytrain["participant_index"].to_numpy(),
)
oof_champion = pairwise_postprocessor.transform(
    probabilities=oof_hmm_only,
    dialog_ids=ytrain["dialog_id"].to_numpy(),
    participant_indices=ytrain["participant_index"].to_numpy(),
)
pl.DataFrame(
    {
        "stage": [
            "Meta-Stacker Raw",
            "+ HMM Chrono Smoothing",
            "+ Pairwise Calibration Only",
            "Champion (HMM + Pairwise)",
        ],
        "val_logloss": [
            round(float(log_loss(y_train, oof_meta_raw)), 5),
            round(float(log_loss(y_train, oof_hmm_only)), 5),
            round(float(log_loss(y_train, oof_pairwise_only)), 5),
            round(float(log_loss(y_train, oof_champion)), 5),
        ],
        "val_roc_auc": [
            round(float(roc_auc_score(y_train, oof_meta_raw)), 5),
            round(float(roc_auc_score(y_train, oof_hmm_only)), 5),
            round(float(roc_auc_score(y_train, oof_pairwise_only)), 5),
            round(float(roc_auc_score(y_train, oof_champion)), 5),
        ],
    }
)

stage,val_logloss,val_roc_auc
str,f64,f64
"""Meta-Stacker Raw""",0.16084,0.98318
"""+ HMM Chrono Smoothing""",0.1423,0.98684
"""+ Pairwise Calibration Only""",0.15967,0.98376
"""Champion (HMM + Pairwise)""",0.14288,0.98696


## Исследование веса марковского сглаживания

Исследуем чувствительность валидационного LogLoss к силе влияния марковского сглаживания (сетка весов блендинга от 0.0 до 1.0).

In [7]:
weights_grid = [round(float(x), 2) for x in np.linspace(0.0, 1.0, 101)]
grid_rows = []
for w_val in weights_grid:
    p_sm = hmm_postprocessor.transform(
        probabilities=oof_meta_raw,
        dialog_ids=ytrain["dialog_id"].to_numpy(),
        participant_indices=ytrain["participant_index"].to_numpy(),
        dialog_order=train_dialog_order,
        blend_weight=w_val,
    )
    p_cal = pairwise_postprocessor.transform(
        probabilities=p_sm,
        dialog_ids=ytrain["dialog_id"].to_numpy(),
        participant_indices=ytrain["participant_index"].to_numpy(),
    )
    grid_rows.append(
        {
            "blend_weight": w_val,
            "logloss_smoothed": round(float(log_loss(y_train, p_sm)), 5),
            "logloss_calibrated": round(float(log_loss(y_train, p_cal)), 5),
            "roc_auc": round(float(roc_auc_score(y_train, p_cal)), 5),
        }
    )

df_grid = pl.DataFrame(grid_rows).sort("logloss_calibrated")
best_w = float(df_grid.row(0, named=True)["blend_weight"])
df_grid.head(10)

blend_weight,logloss_smoothed,logloss_calibrated,roc_auc
f64,f64,f64,f64
0.68,0.14205,0.14261,0.98688
0.69,0.14206,0.14261,0.98672
0.67,0.14205,0.14262,0.98705
0.7,0.14207,0.14262,0.98673
0.71,0.1421,0.14263,0.98673
0.66,0.14206,0.14264,0.98703
0.65,0.14208,0.14265,0.98703
0.72,0.14214,0.14267,0.98674
0.64,0.1421,0.14268,0.98702


## Расчет финального сабмишна и логирование в MLflow

Применяем оптимизированный HMM-постпроцессинг и парную диалоговую калибровку к предсказаниям на тестовой выборке и фиксируем чемпионские метрики в MLflow Tracking.

In [8]:
oof_hmm_best = hmm_postprocessor.transform(
    probabilities=oof_meta_raw,
    dialog_ids=ytrain["dialog_id"].to_numpy(),
    participant_indices=ytrain["participant_index"].to_numpy(),
    dialog_order=train_dialog_order,
    blend_weight=best_w,
)
oof_champion = pairwise_postprocessor.transform(
    probabilities=oof_hmm_best,
    dialog_ids=ytrain["dialog_id"].to_numpy(),
    participant_indices=ytrain["participant_index"].to_numpy(),
)

test_hmm = hmm_postprocessor.transform(
    probabilities=test_meta_raw,
    dialog_ids=ytest["dialog_id"].to_numpy(),
    participant_indices=ytest["participant_index"].to_numpy(),
    dialog_order=test_dialog_order,
    blend_weight=best_w,
)
test_final_probs = pairwise_postprocessor.transform(
    probabilities=test_hmm,
    dialog_ids=ytest["dialog_id"].to_numpy(),
    participant_indices=ytest["participant_index"].to_numpy(),
)

sub_df = pl.DataFrame({"ID": ytest["ID"].to_numpy(), "is_bot": test_final_probs})

champ_loss = float(log_loss(y_train, oof_champion))
d_lens = np.array([len(train_dialogs.get(str(d), [])) for d in ytrain["dialog_id"]])
loss_short = float(log_loss(y_train[d_lens <= 2], oof_champion[d_lens <= 2]))
loss_long = float(log_loss(y_train[d_lens > 2], oof_champion[d_lens > 2]))

hmm_feature_names = [
    "meta_stacker_raw_prob",
    "hmm_forward_backward_posterior",
    "pairwise_calibrated_prob",
]

tracker_champ = (
    ExperimentTrackerBuilder(run_name="hmm_champion_master", experiment_name="bot-detection")
    .set_description(
        "Финальный чемпионский пайплайн: Level-2 Meta-Stacker + турнирный HMM (Forward-Backward) + парная калибровка."
    )
    .set_dataset(
        data=pl.DataFrame(
            {
                "meta_stacker_raw_prob": oof_meta_raw,
                "hmm_forward_backward_posterior": oof_hmm_best,
                "pairwise_calibrated_prob": oof_champion,
            }
        ),
        name="train_champion_postprocessed",
        targets=y_train,
    )
    .set_params(
        model="HMMPostprocessor",
        model_family="HMM_Pairwise_MetaStacker_3Trees",
        level1_models="CatBoost,LightGBM,XGBoost,RuBERT",
        cv_method="GroupKFold(n_splits=5)",
        optimizer="GridSearch",
        n_trials=len(weights_grid),
        hmm_blend_weight=best_w,
        alpha_laplace=1.0,
        pairwise_max_sum=1.0,
        smoothing="forward_backward",
        order="original_chronological",
        seed=42,
    )
    .require_feature_tree(False)
    .defer_metrics()
    .build()
)

with tracker_champ:
    best_so_far = float("inf")
    for step_idx, row in enumerate(grid_rows):
        val = float(row["logloss_calibrated"])
        best_so_far = min(best_so_far, val)
        mlflow.log_metric("optuna_trial_val_logloss", val, step=step_idx)
        mlflow.log_metric("optuna_best_val_logloss", best_so_far, step=step_idx)

    fig_blend, ax_blend = plt.subplots(figsize=(7, 4.5))
    w_arr = [r["blend_weight"] for r in grid_rows]
    l_sm = [r["logloss_smoothed"] for r in grid_rows]
    l_cal = [r["logloss_calibrated"] for r in grid_rows]
    ax_blend.plot(w_arr, l_sm, label="HMM Smoothed LogLoss", linestyle="--", color="#3498db")
    ax_blend.plot(w_arr, l_cal, label="HMM + Pairwise Calibrated LogLoss", linewidth=2, color="#2ecc71")
    ax_blend.axvline(best_w, color="#e74c3c", linestyle=":", label=f"Best w={best_w:.2f} ({champ_loss:.5f})")
    ax_blend.set_title("HMM Blend Weight Tuning Curve (5-Fold OOF)")
    ax_blend.set_xlabel("blend_weight (w)")
    ax_blend.set_ylabel("CV LogLoss")
    ax_blend.grid(True, alpha=0.3)
    ax_blend.legend()
    fig_blend.tight_layout()
    mlflow.log_figure(fig_blend, "graphics/hmm_blend_curve.png")
    plt.close(fig_blend)

    tracker_champ.log_metrics(
        val_logloss=float(champ_loss),
        val_logloss_raw=float(log_loss(y_train, oof_hmm_best)),
        val_roc_auc=float(roc_auc_score(y_train, oof_champion)),
        val_accuracy=float(accuracy_score(y_train, (oof_champion >= 0.5).astype(int))),
        val_f1=float(f1_score(y_train, (oof_champion >= 0.5).astype(int))),
        val_logloss_short=loss_short,
        val_logloss_long=loss_long,
        pairwise_consistency_rate=float(np.mean(oof_champion[0::2] + oof_champion[1::2] <= 1.0 + 1e-5)),
    )
    tracker_champ.log_feature_tree(
        {
            "architecture": "HMMPostprocessor",
            "features": [{"name": f, "importance": 1.0 / len(hmm_feature_names)} for f in hmm_feature_names],
        }
    )
    tracker_champ.log_all_graphics(y_true=y_train, y_pred_proba=oof_champion)
    tracker_champ.log_submission(sub_df, filename="submission.csv")

pl.DataFrame(
    {
        "final_pipeline": ["Champion HMM + Pairwise Postprocessor"],
        "best_blend_weight": [best_w],
        "val_logloss": [round(float(champ_loss), 5)],
        "val_roc_auc": [round(float(roc_auc_score(y_train, oof_champion)), 5)],
        "status": ["tournament_submission_generated"],
    }
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run hmm_champion_master at: http://127.0.0.1:5000/#/experiments/7/runs/1924e5b7f49b40e2a81eb3b29795aa5c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


final_pipeline,best_blend_weight,val_logloss,val_roc_auc,status
str,f64,f64,f64,str
"""Champion HMM + Pairwise Postpr…",0.68,0.14261,0.98688,"""tournament_submission_generate…"


## Выводы по марковской постобработке

- Модель переходов учитывает хронологическую автокорреляцию сессий в турнирной сетке соревнований.
- Вероятность повторного появления бота в следующей сессии существенно выше случайного распределения.
- Совмещение марковского сглаживания Forward-Backward и внутридиалоговой парной нормализации обеспечивает наилучшее качество на валидации.
- Сгенерирован и верифицирован финальный сабмишн `submission.csv`.